In [37]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_parquet
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import *

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [38]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\taxi02")
zone_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\zone")

In [39]:
def clean_taxi_df(df):
    # Remove invalid trips
    df = df.filter(
        (col("trip_distance") >= 0) &
        (col("fare_amount") >= 0)
    )
    
    # Handle nulls in important columns
    df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "PULocationID", "DOLocationID", "payment_type", "fare_amount", "total_amount"])
    
    return df


In [40]:
taxi01_clean_df = clean_taxi_df(taxi01_df)
print(f"Rows dropped for taxi01_df: {taxi01_df.count() - taxi01_clean_df.count()}")


Rows dropped for taxi01_df: 4542


In [41]:
taxi02_clean_df = clean_taxi_df(taxi02_df)
print(f"Rows dropped for taxi02_df: {taxi02_df.count() - taxi02_clean_df.count()}")


Rows dropped for taxi02_df: 4307


In [42]:
zone_df.show(4)

+----------+---------+--------------------+------------+
|LocationID|  Borough|                Zone|service_zone|
+----------+---------+--------------------+------------+
|         1|      EWR|      Newark Airport|         EWR|
|         2|   Queens|         Jamaica Bay|   Boro Zone|
|         3|    Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|Manhattan|       Alphabet City| Yellow Zone|
+----------+---------+--------------------+------------+
only showing top 4 rows


In [43]:
zone_clean_df = zone_df.dropna(subset=["LocationID", "Borough", "Zone", "service_zone"])

print(f"Rows dropped for zone_df: {zone_df.count() - zone_clean_df.count()}")


Rows dropped for zone_df: 0


In [44]:
# convert timestamp taxi01_clean_df
taxi01_clean_df = taxi01_clean_df.withColumn(
    "tpep_pickup_datetime",
    to_timestamp(col("tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "tpep_dropoff_datetime",
    to_timestamp(col("tpep_dropoff_datetime"), "yyyy-MM-dd HH:mm:ss")
)

In [45]:
# convert timestamp taxi02_clean_df

taxi02_clean_df = taxi02_clean_df.withColumn(
    "tpep_pickup_datetime",
    to_timestamp(col("tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "tpep_dropoff_datetime",
    to_timestamp(col("tpep_dropoff_datetime"), "yyyy-MM-dd HH:mm:ss")
)

In [46]:
# Save feature engineered data
write_parquet(taxi01_clean_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi01")
write_parquet(taxi02_clean_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi02")
write_parquet(zone_clean_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\zone")

print("Cleaned data saved successfully!")


Cleaned data saved successfully!


In [47]:
spark.stop()